# Batch xenofobia — Parte 2: Chequear y bajar

Este notebook:
1. Consulta el estado de todos los batches (sin polling, una sola vez)
2. Descarga los que están `completed`
3. Los combina y une con el CSV original

**Actualizá `BATCH_REGISTRY` con los IDs que imprimió el notebook de subida antes de ejecutar.**

In [2]:
import os, json
from pathlib import Path
import pandas as pd
from openai import OpenAI

client = OpenAI(api_key=os.environ['OPENAI_API_KEY'])
print('Cliente OK')

Cliente OK


In [3]:
INPUT_CSV           = 'DNU_data/tweets_clasificados.csv'
JSONL_DIR           = Path('outs_temp')
OUTPUT_CSV          = 'outputs/dnu_gpt_xenofobia_final.csv'
PRICE_INPUT_PER_1M  = 0.40
PRICE_OUTPUT_PER_1M = 1.60

In [4]:
# ── Actualizá acá con los IDs que imprimió el notebook de subida ──────────
# None = todavía no enviado / fallido, actualizá después de subir
BATCH_REGISTRY = {
    1: 'batch_69d055d51dac8190b74033f1192fe9b9',
     2: 'batch_69d055ff99ac819092b4cdbc0316d286',
     3: 'batch_69d0562a9bd081908c953e14b591dcf5',
     4: 'batch_69d05655b2b0819089abbed186402091',
     5: 'batch_69d166d415d88190b4b495b3c7f82008',
     6: 'batch_69d166de289c8190b8e294d1a2011c76',
}

In [5]:
VALID_LABELS = {'XENOPHOBIC', 'NOT_XENOPHOBIC'}

def extract_label(raw):
    if raw is None:
        return None
    raw = str(raw).strip()
    for candidate in [raw, raw[raw.find('{'):raw.rfind('}') + 1]]:
        try:
            parsed = json.loads(candidate)
            if isinstance(parsed, dict) and 'label' in parsed:
                return str(parsed['label']).strip().upper()
        except Exception:
            pass
    return None

def parse_result_line(line):
    obj = json.loads(line)
    custom_id = obj.get('custom_id', '')
    row_id = int(custom_id.replace('row-', '')) if custom_id.startswith('row-') else None

    raw_content = prompt_t = completion_t = total_t = status_code = None
    error_msg = None

    response = obj.get('response')
    if response:
        status_code = response.get('status_code')
        body  = response.get('body', {})
        usage = body.get('usage', {})
        prompt_t     = usage.get('prompt_tokens')
        completion_t = usage.get('completion_tokens')
        total_t      = usage.get('total_tokens')
        choices = body.get('choices', [])
        if choices:
            raw_content = choices[0].get('message', {}).get('content')
    if obj.get('error'):
        error_msg = str(obj['error'])

    label = extract_label(raw_content)
    return {
        'row_id':                  row_id,
        'batch_label':             label,
        'batch_is_xenophobic':     1 if label == 'XENOPHOBIC' else (0 if label == 'NOT_XENOPHOBIC' else None),
        'batch_raw_response':      raw_content,
        'batch_invalid_label':     label if label not in VALID_LABELS | {None} else None,
        'batch_prompt_tokens':     prompt_t,
        'batch_completion_tokens': completion_t,
        'batch_total_tokens':      total_t,
        'batch_status_code':       status_code,
        'batch_error':             error_msg,
    }

print('Helpers OK')

Helpers OK


---
## Paso 1 — Chequear estado (una sola consulta, sin polling)

In [8]:
print(f'{"Parte":>6}  {"Batch ID":47}  {"Estado":<15}  Completados/Total')
print('-' * 92)

batch_status = {}
completed_parts = []
pending_parts   = []
missing_parts   = []

for part, batch_id in sorted(BATCH_REGISTRY.items()):
    if batch_id is None:
        batch_status[part] = 'not_submitted'
        missing_parts.append(part)
        print(f'{part:>6}  {"(sin enviar)":47}  not_submitted')
    else:
        st  = client.batches.retrieve(batch_id)
        req = st.request_counts
        batch_status[part] = st.status
        print(f'{part:>6}  {batch_id:47}  {st.status:<15}  {req.completed}/{req.total}')
        if st.status == 'completed':
            completed_parts.append(part)
        elif st.status in ('failed', 'expired', 'cancelled'):
            missing_parts.append(part)
        else:  # in_progress, validating, etc.
            pending_parts.append(part)

print()
print(f'  Completados y listos para bajar : {completed_parts}')
print(f'  Todavía en proceso              : {pending_parts}')
print(f'  Fallidos / sin enviar (ir a nb1): {missing_parts}')

 Parte  Batch ID                                         Estado           Completados/Total
--------------------------------------------------------------------------------------------
     1  batch_69d055d51dac8190b74033f1192fe9b9           completed        50000/50000
     2  batch_69d055ff99ac819092b4cdbc0316d286           completed        50000/50000
     3  batch_69d0562a9bd081908c953e14b591dcf5           completed        50000/50000
     4  batch_69d05655b2b0819089abbed186402091           completed        50000/50000
     5  batch_69d166d415d88190b4b495b3c7f82008           completed        50000/50000
     6  batch_69d166de289c8190b8e294d1a2011c76           completed        9752/9752

  Completados y listos para bajar : [1, 2, 3, 4, 5, 6]
  Todavía en proceso              : []
  Fallidos / sin enviar (ir a nb1): []


---
## Paso 2 — Descargar los completados

Solo baja los que están `completed`. Si el archivo ya existe en disco, lo saltea.

In [9]:
JSONL_DIR.mkdir(exist_ok=True)
downloaded = {}

for part in sorted(completed_parts):
    batch_id    = BATCH_REGISTRY[part]
    result_path = JSONL_DIR / f'batch_results_xenophobia_part_{part}.jsonl'

    if result_path.exists():
        kb = result_path.stat().st_size / 1024
        print(f'Parte {part:2d}: ya en disco  ({kb:.0f} KB)  {result_path}')
    else:
        st      = client.batches.retrieve(batch_id)
        content = client.files.content(st.output_file_id)
        with open(result_path, 'wb') as fout:
            fout.write(content.read())
        kb = result_path.stat().st_size / 1024
        print(f'Parte {part:2d}: descargado   ({kb:.0f} KB)  {result_path}')

    downloaded[part] = result_path

total_parts = len(BATCH_REGISTRY)
print()
print(f'Total descargadas: {len(downloaded)}/{total_parts}  -  partes {sorted(downloaded.keys())}')
if pending_parts or missing_parts:
    print(f'Faltan: pendientes={pending_parts}  sin enviar/fallidos={missing_parts}')
    print('Volvé a ejecutar este notebook cuando estén listos, o usá el notebook de subida.')

Parte  1: descargado   (42081 KB)  outs_temp\batch_results_xenophobia_part_1.jsonl
Parte  2: descargado   (42089 KB)  outs_temp\batch_results_xenophobia_part_2.jsonl
Parte  3: descargado   (42153 KB)  outs_temp\batch_results_xenophobia_part_3.jsonl
Parte  4: descargado   (42161 KB)  outs_temp\batch_results_xenophobia_part_4.jsonl
Parte  5: descargado   (42174 KB)  outs_temp\batch_results_xenophobia_part_5.jsonl
Parte  6: descargado   (8226 KB)  outs_temp\batch_results_xenophobia_part_6.jsonl

Total descargadas: 6/6  -  partes [1, 2, 3, 4, 5, 6]


---
## Paso 3 — Parsear, unir con el CSV original y guardar

In [13]:
if not downloaded:
    print('No hay resultados descargados todavía. Esperá a que completen los batches.')
else:
    # Parsear todos los JSONL descargados
    rows = []
    for part in sorted(downloaded.keys()):
        n_prev = len(rows)
        with open(downloaded[part], 'r', encoding='utf-8') as fin:
            for line in fin:
                rows.append(parse_result_line(line))
        print(f'Parte {part:2d}: {len(rows) - n_prev:>6,} filas  (total: {len(rows):,})')

    results_df = pd.DataFrame(rows).sort_values('row_id').reset_index(drop=True)
    print(f'\nTotal resultados: {len(results_df):,}')

    # Cargar CSV original y hacer merge
    df = pd.read_csv(INPUT_CSV)
    df.index.name = 'row_id'
    df = df.reset_index()

    final_df = df.merge(results_df, on='row_id', how='left')

    # Guardar
    Path(OUTPUT_CSV).parent.mkdir(parents=True, exist_ok=True)
    final_df.to_csv(OUTPUT_CSV, index=False)

    print(f'\nCSV guardado en: {OUTPUT_CSV}')
    print(f'Filas totales  : {len(final_df):,}')
    print(f'Clasificadas   : {final_df["batch_label"].notna().sum():,}')
    print(f'Sin clasificar : {final_df["batch_label"].isna().sum():,}  (partes pendientes)')

Parte  1: 50,000 filas  (total: 50,000)
Parte  2: 50,000 filas  (total: 100,000)
Parte  3: 50,000 filas  (total: 150,000)
Parte  4: 50,000 filas  (total: 200,000)
Parte  5: 50,000 filas  (total: 250,000)
Parte  6:  9,752 filas  (total: 259,752)

Total resultados: 259,752

CSV guardado en: outputs/dnu_gpt_xenofobia_final.csv
Filas totales  : 259,752
Clasificadas   : 259,752
Sin clasificar : 0  (partes pendientes)


In [14]:
if not downloaded:
    print('Sin datos para resumir.')
else:
    total   = len(final_df)
    labeled = final_df['batch_label'].notna().sum()

    print('=' * 50)
    print(f'Filas totales        : {total:>10,}')
    print(f'Clasificadas         : {labeled:>10,}  ({labeled/total*100:.1f}%)')
    print()
    print(final_df['batch_label'].value_counts(dropna=False).to_string())
    print()

    prompt_sum     = pd.to_numeric(final_df['batch_prompt_tokens'],     errors='coerce').sum()
    completion_sum = pd.to_numeric(final_df['batch_completion_tokens'], errors='coerce').sum()
    cost = (prompt_sum / 1e6 * PRICE_INPUT_PER_1M + completion_sum / 1e6 * PRICE_OUTPUT_PER_1M) * 0.5

    print(f'Prompt tokens total  : {int(prompt_sum):>10,}')
    print(f'Completion tokens    : {int(completion_sum):>10,}')
    print(f'Costo batch real     : {cost:>10.4f} USD')
    print('=' * 50)

Filas totales        :    259,752
Clasificadas         :    259,752  (100.0%)

batch_label
NOT_XENOPHOBIC    215408
XENOPHOBIC         44344

Prompt tokens total  : 48,811,049
Completion tokens    :  2,812,928
Costo batch real     :    12.0126 USD
